# Market Creator Activity Dashboard

**Objective:** Monitor and analyze market creator activity on Gnosis Chain.

## How to run

1. Ensure you have the root `.env` file with `GNOSIS_RPC` set.
2. Install dependencies: `poetry install` from the repo root.
3. Run all cells top-to-bottom.

In [ ]:
import sys
import os

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Allow imports from repo root
sys.path.insert(0, os.path.abspath(".."))

from market_creator_info import NATIVE_TOKEN, WXDAI_ADDRESS, get_all_creators_info
from omen_markets import MarketState

# Canonical icons for each market state — use these everywhere in the notebook
MARKET_STATE_ICONS = {
    MarketState.OPEN: "\U0001f4e5",          # 📥 inbox tray
    MarketState.PENDING: "\U0001f4cb",       # 📋 clipboard
    MarketState.FINALIZING: "\u23f3",        # ⏳ hourglass
    MarketState.ARBITRATING: "\u2696\ufe0f", # ⚖️ balance scale
    MarketState.CLOSED: "\U0001f512",        # 🔒 lock
    MarketState.UNKNOWN: "\u2753",           # ❓ question mark
}

## Configuration

Define market creators here. Each entry maps a label to a config dict.

Thresholds are per-token-address for `safe` and `owner` separately.
Use `NATIVE_TOKEN` (zero address) for native xDAI, or any ERC-20 contract address.

In [ ]:
TOKEN_LABELS = {
    NATIVE_TOKEN: "xDAI",
    WXDAI_ADDRESS: "wxDAI",
}

MARKET_CREATORS = {
    "qs": {
        "name": "QS Market Creator",
        "safe_contract_address": "0x89c5cc945dd550BcFfb72Fe42BfF002429F46Fec",
        "markets_to_approve_per_day": 15,
        "thresholds": {
            "safe": {
                NATIVE_TOKEN: 1.0,
                WXDAI_ADDRESS: 50.0,
            },
            "owner": {
                NATIVE_TOKEN: 0.5,
                WXDAI_ADDRESS: 0,
            },
        },
    },
    "pearl": {
        "name": "Pearl Market Creator",
        "safe_contract_address": "0xFfc8029154ECD55ABED15BD428bA596E7D23f557",
        "markets_to_approve_per_day": 8,
        "thresholds": {
            "safe": {
                NATIVE_TOKEN: 1.0,
                WXDAI_ADDRESS: 50.0,
            },
            "owner": {
                NATIVE_TOKEN: 0.5,
                WXDAI_ADDRESS: 0,
            },
        },
    },
}

## Balances

Fetch xDAI and wxDAI balances for each market creator's Safe and owner/signer addresses.
A red circle indicates a balance below the configured threshold.

In [ ]:
creators = get_all_creators_info(MARKET_CREATORS)

# Build a lookup from name -> config for threshold access
_cfg_by_name = {cfg["name"]: cfg for cfg in MARKET_CREATORS.values()}

def _alert(value: float, threshold: float) -> str:
    """Return a red dot if below threshold, green dot if OK (skip if threshold is 0)."""
    icon = "\U0001f534" if (threshold > 0 and value < threshold) else "\U0001f7e2"
    return f"{value:.4f} {icon}"

rows = []
for c in creators:
    cfg = _cfg_by_name[c.label]
    safe_thresholds = cfg["thresholds"]["safe"]
    owner_thresholds = cfg["thresholds"]["owner"]

    # Safe row
    row = {
        "Label": c.label,
        "Type": "Safe",
        "Address": c.safe_contract_address,
        "Sig. Threshold": f"{c.threshold}/{len(c.owners)}",
    }
    for token_addr, threshold in safe_thresholds.items():
        label = TOKEN_LABELS.get(token_addr, token_addr[:10])
        balance = c.balances[c.safe_contract_address].get(token_addr, 0)
        row[label] = _alert(balance, threshold)
    rows.append(row)

    # Owner rows
    for owner in c.owners:
        row = {
            "Label": c.label,
            "Type": "Owner/Signer",
            "Address": owner,
            "Sig. Threshold": "",
        }
        for token_addr, threshold in owner_thresholds.items():
            label = TOKEN_LABELS.get(token_addr, token_addr[:10])
            balance = c.balances[owner].get(token_addr, 0)
            row[label] = _alert(balance, threshold)
        rows.append(row)

df_balances = pd.DataFrame(rows)

# Right-align balance columns so the status dots line up
balance_cols = list(TOKEN_LABELS.values())
col_styles = [
    {"selector": f"td.col{df_balances.columns.get_loc(col)}", "props": "text-align: right;"}
    for col in balance_cols if col in df_balances.columns
]
display(HTML(df_balances.style.set_table_styles(col_styles).hide(axis="index").to_html()))

## Markets

Fetch all markets from the Omen subgraph and display:
1. Current state breakdown (open, pending, finalizing, arbitrating, closed)
2. Daily history of markets opened, closed (answered), and resolved (finalized) over the last N days

In [ ]:
from market_activity import fetch_all_markets, market_state_summary, daily_activity, answer_distribution

markets_by_creator = fetch_all_markets(MARKET_CREATORS)

### Markets by state

In [ ]:
df_states = market_state_summary(markets_by_creator, MARKET_CREATORS)
display(HTML(df_states.to_html()))

### Markets daily history

In [ ]:
from datetime import datetime, timedelta, timezone

# Number of recent days for which statistics are displayed (activity, answer distribution, mech results)
STATS_PERIOD_DAYS = 14

now_utc = datetime.now(timezone.utc)
period_start = (now_utc - timedelta(days=STATS_PERIOD_DAYS - 1)).replace(hour=0, minute=0, second=0)
period_end = now_utc

display(HTML(
    f"<b>Study period:</b> {period_start.strftime('%Y-%m-%d %H:%M UTC')} — "
    f"{period_end.strftime('%Y-%m-%d %H:%M UTC')} ({STATS_PERIOD_DAYS} days)"
))

df_opened, df_responded, df_closed = daily_activity(
    markets_by_creator, MARKET_CREATORS, n_days=STATS_PERIOD_DAYS
)

# Build expected markets per day lookup: creator name -> expected count
_expected_per_day = {
    cfg["name"]: cfg["markets_to_approve_per_day"]
    for cfg in MARKET_CREATORS.values()
}

def _format_opened(val, creator_name):
    """Red dot if value differs from expected markets_to_approve_per_day."""
    expected = _expected_per_day.get(creator_name, None)
    if expected is not None and val != expected:
        return f"{val} \U0001f534"
    return f"{val} \U0001f7e2"

# One table per creator
_open_icon = MARKET_STATE_ICONS[MarketState.OPEN]
_closed_icon = MARKET_STATE_ICONS[MarketState.CLOSED]

for key, cfg in MARKET_CREATORS.items():
    name = cfg["name"]
    expected = cfg.get("markets_to_approve_per_day", "?")

    dates = df_opened.index.sort_values(ascending=False)
    rows = []
    for d in dates:
        opened_val = df_opened.at[d, name] if name in df_opened.columns else 0
        responded_val = df_responded.at[d, name] if name in df_responded.columns else 0
        closed_val = df_closed.at[d, name] if name in df_closed.columns else 0
        rows.append({
            "Date": d,
            f"{_open_icon} Open": _format_opened(opened_val, name),
            "Responded": int(responded_val),
            f"{_closed_icon} Closed": int(closed_val),
        })

    display(HTML(f"<h4>{name} (expected open/day: {expected})</h4>"))
    display(HTML(pd.DataFrame(rows).to_html(escape=False, index=False)))

### Closed markets answer analysis

In [ ]:
from collections import Counter
from market_activity import _ts_to_date

_period_label = (
    f"from {period_start.strftime('%Y-%m-%d %H:%M UTC')} "
    f"to {period_end.strftime('%Y-%m-%d %H:%M UTC')}"
)

ANSWER_COLORS = {"Yes": "#4CAF50", "No": "#F44336", "Invalid": "#9E9E9E", "N/A": "#BDBDBD"}


def _plot_answer_distribution(dist, subtitle):
    """Plot answer distribution pie charts for each creator."""
    n = len(dist)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, (name, counts) in zip(axes, dist.items()):
        if not counts:
            ax.set_title(f"{name}\n(no closed markets)")
            ax.axis("off")
            continue
        labels = list(counts.keys())
        values = list(counts.values())
        colors = [ANSWER_COLORS.get(l, "#607D8B") for l in labels]
        total = sum(values)
        ax.pie(
            values, labels=labels, colors=colors,
            autopct=lambda pct, t=total: f"{pct:.1f}%\n({int(round(pct * t / 100))})",
            startangle=90, textprops={"fontsize": 10},
        )
        ax.set_title(f"{name}\n(n={total})", fontsize=12, fontweight="bold")
    fig.suptitle(f"Answer Distribution\n{subtitle}", fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout()
    plt.show()


def _get_distinct_answer_counts(markets_by_creator, creators, n_days=None):
    """Count distinct answer values per closed market."""
    today = datetime.now(timezone.utc).date()
    date_set = None
    if n_days is not None:
        date_set = set(today - timedelta(days=i) for i in range(n_days))
    result = {}
    for key, markets in markets_by_creator.items():
        name = creators[key]["name"]
        counts = []
        for m in markets:
            answer_ts = (m.get("question") or {}).get("currentAnswerTimestamp")
            if not answer_ts:
                continue
            if date_set is not None and _ts_to_date(answer_ts) not in date_set:
                continue
            answers = (m.get("question") or {}).get("answers") or []
            counts.append(len(answers))
        result[name] = counts
    return result


def _display_distinct_answer_table(counts_by_creator, subtitle):
    """Display a summary table of distinct answer counts per creator."""
    all_values = sorted(set(v for vals in counts_by_creator.values() for v in vals))
    if not all_values:
        display(HTML(f"<h4>Number of distinct answers in closed markets — {subtitle}</h4>"))
        display(HTML("<p><i>No closed markets.</i></p>"))
        return

    rows = []
    for name, vals in counts_by_creator.items():
        total = len(vals)
        if total == 0:
            continue
        counter = Counter(vals)
        row = {"Creator": f"{name} (n={total})"}
        for v in all_values:
            count = counter.get(v, 0)
            pct = count / total * 100
            row[f"{v} answer{'s' if v != 1 else ''}"] = f"{count} ({pct:.1f}%)"
        rows.append(row)

    display(HTML(f"<h4>Number of distinct answers in closed markets — {subtitle}</h4>"))
    display(HTML(pd.DataFrame(rows).to_html(escape=False, index=False)))


_closed_period = f"Markets closed {_period_label}"

# 1. Answer distribution (study period)
dist_period = answer_distribution(markets_by_creator, MARKET_CREATORS, n_days=STATS_PERIOD_DAYS)
_plot_answer_distribution(dist_period, _closed_period)

# 2. Answer distribution (overall)
dist_overall = answer_distribution(markets_by_creator, MARKET_CREATORS, n_days=365 * 10)
_plot_answer_distribution(dist_overall, "All closed markets")

# 3. Number of distinct answers (study period + overall)
counts_period = _get_distinct_answer_counts(markets_by_creator, MARKET_CREATORS, n_days=STATS_PERIOD_DAYS)
_display_distinct_answer_table(counts_period, _closed_period)

counts_overall = _get_distinct_answer_counts(markets_by_creator, MARKET_CREATORS)
_display_distinct_answer_table(counts_overall, "All closed markets")

### Audit analysis

Comparison of market answers against AI audit predictions (OpenAI, Grok, Gemini).
Audit data is loaded from `fpmm_audits.json` (generated externally).

In [ ]:
import json as _json
from omen_markets import (
    INVALID_ANSWER_HEX, get_market_current_answer as _get_answer,
)

# Load audits
_AUDITS_PATH = "fpmm_audits.json"
try:
    with open(_AUDITS_PATH, "r", encoding="UTF-8") as _f:
        _audit_data = _json.load(_f).get("fpmmAudits", {})
except FileNotFoundError:
    _audit_data = {}

# Build a flat lookup: market_id -> market dict (across all creators)
_markets_by_id = {}
for _key, _mlist in markets_by_creator.items():
    for _m in _mlist:
        _markets_by_id[_m["id"]] = _m


def _audit_stats(market, market_audits):
    """Return (matching, valid, total) for a single market's audits."""
    answer = _get_answer(market).lower()
    outcomes = [o.lower() for o in (market.get("question") or {}).get("outcomes", [])]
    total = matching = valid = 0
    for a in market_audits.values():
        if not isinstance(a, dict):
            continue
        resp_answer = (a.get("response") or {}).get("answer")
        if not isinstance(resp_answer, str):
            continue
        total += 1
        if resp_answer.lower() not in outcomes:
            continue
        valid += 1
        if resp_answer.lower() == answer:
            matching += 1
    return matching, valid, total


# Gather per-market audit stats
_rows = []
for fpmm_id, audits in _audit_data.items():
    market = _markets_by_id.get(fpmm_id)
    if not market:
        continue
    matching, valid, total = _audit_stats(market, audits)
    if total == 0:
        continue
    current_answer = market.get("currentAnswer", "")
    is_invalid = isinstance(current_answer, str) and current_answer.lower() == INVALID_ANSWER_HEX.lower()
    _rows.append({
        "market_id": fpmm_id,
        "matching": matching,
        "valid": valid,
        "total": total,
        "is_invalid": is_invalid,
        "answer": _get_answer(market),
    })

df_audit = pd.DataFrame(_rows)

if len(df_audit) == 0:
    display(HTML("<p><i>No audit data available.</i></p>"))
else:
    n_audited = len(df_audit)
    n_invalid = df_audit["is_invalid"].sum()
    n_valid_markets = n_audited - n_invalid

    # --- Summary table ---
    summary_rows = [
        {"Metric": "Total audited markets", "Value": n_audited},
        {"Metric": "Markets with Invalid answer", "Value": f"{n_invalid} ({n_invalid/n_audited*100:.1f}%)"},
        {"Metric": "Markets with valid answer", "Value": f"{n_valid_markets} ({n_valid_markets/n_audited*100:.1f}%)"},
    ]
    display(HTML("<h4>Audit overview</h4>"))
    display(HTML(pd.DataFrame(summary_rows).to_html(escape=False, index=False)))

    # Filter to valid-answer markets for the rest of the analysis
    df_valid = df_audit[~df_audit["is_invalid"]].copy()

    if len(df_valid) > 0:
        # --- Audit counts distribution ---
        display(HTML("<h4>Number of audits per market</h4>"))
        total_dist = df_valid["total"].value_counts().sort_index()
        rows_dist = []
        for n_audits, count in total_dist.items():
            rows_dist.append({
                "# Audits": n_audits,
                "Markets": count,
                "%": f"{count/len(df_valid)*100:.1f}%",
            })
        display(HTML(pd.DataFrame(rows_dist).to_html(escape=False, index=False)))

        # --- Valid audits distribution ---
        display(HTML("<h4>Number of valid audits per market</h4>"))
        valid_dist = df_valid["valid"].value_counts().sort_index()
        rows_vdist = []
        for n_valid, count in valid_dist.items():
            rows_vdist.append({
                "# Valid audits": n_valid,
                "Markets": count,
                "%": f"{count/len(df_valid)*100:.1f}%",
            })
        display(HTML(pd.DataFrame(rows_vdist).to_html(escape=False, index=False)))

        # --- Matching audits breakdown by valid audit count ---
        display(HTML("<h4>Matching audits by number of valid audits</h4>"))

        # Table: for each valid count, show how many markets have 0, 1, 2, ... matching
        valid_values = sorted(df_valid["valid"].unique())
        breakdown_rows = []
        for v in valid_values:
            if v == 0:
                continue
            subset = df_valid[df_valid["valid"] == v]
            n_subset = len(subset)
            match_counts = subset["matching"].value_counts().sort_index()
            row = {"Valid audits": f"{v} (n={n_subset})"}
            for m in range(v + 1):
                count = match_counts.get(m, 0)
                pct = count / n_subset * 100 if n_subset > 0 else 0
                row[f"{m} match"] = f"{count} ({pct:.1f}%)"
            breakdown_rows.append(row)
        if breakdown_rows:
            display(HTML(pd.DataFrame(breakdown_rows).fillna("").to_html(escape=False, index=False)))

        # Grouped bar chart: one group per "# valid audits", bars for each matching count
        df_with_valid = df_valid[df_valid["valid"] > 0].copy()
        if len(df_with_valid) > 0:
            valid_values_plot = sorted(df_with_valid["valid"].unique())
            max_valid = max(valid_values_plot)
            match_levels = list(range(max_valid + 1))

            fig, ax = plt.subplots(figsize=(10, 5))
            bar_width = 0.8 / (max_valid + 1)
            match_colors = ["#F44336", "#FF9800", "#8BC34A", "#4CAF50", "#2E7D32"]

            for i, m_val in enumerate(match_levels):
                x_positions = []
                heights = []
                for j, v_val in enumerate(valid_values_plot):
                    if m_val > v_val:
                        continue
                    subset = df_with_valid[df_with_valid["valid"] == v_val]
                    count = len(subset[subset["matching"] == m_val])
                    x_positions.append(j + i * bar_width)
                    heights.append(count)
                if heights:
                    color = match_colors[min(i, len(match_colors) - 1)]
                    ax.bar(x_positions, heights, width=bar_width, label=f"{m_val} matching",
                           color=color, edgecolor="black", alpha=0.85)

            ax.set_xlabel("Number of valid audits")
            ax.set_ylabel("Number of markets")
            ax.set_xticks([j + bar_width * (max_valid / 2) for j in range(len(valid_values_plot))])
            ax.set_xticklabels([str(v) for v in valid_values_plot])
            ax.legend(title="Matching audits")
            ax.grid(True, alpha=0.3, axis="y")
            ax.set_title(f"Matching audits by number of valid audits\n(n={len(df_with_valid)})",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()

        # --- Pass / Partial / Fail breakdown (only markets with valid > 0) ---
        if len(df_with_valid) > 0:
            def _classify(row):
                if row["matching"] == row["valid"]:
                    return "Pass"
                elif row["matching"] > 0:
                    return "Partial"
                else:
                    return "Fail"

            df_with_valid["result"] = df_with_valid.apply(_classify, axis=1)
            result_counts = df_with_valid["result"].value_counts()

            # Ensure all categories exist
            for cat in ["Pass", "Partial", "Fail"]:
                if cat not in result_counts:
                    result_counts[cat] = 0

            colors = {"Pass": "#4CAF50", "Partial": "#FF9800", "Fail": "#F44336"}
            labels = ["Pass", "Partial", "Fail"]
            values = [result_counts[l] for l in labels]
            total_with_valid = sum(values)

            fig, ax = plt.subplots(figsize=(6, 5))
            nonzero = [(l, v) for l, v in zip(labels, values) if v > 0]
            if nonzero:
                pie_labels, pie_values = zip(*nonzero)
                pie_colors = [colors[l] for l in pie_labels]
                ax.pie(
                    pie_values, labels=pie_labels, colors=pie_colors,
                    autopct=lambda pct, t=total_with_valid: f"{pct:.1f}%\n({int(round(pct * t / 100))})",
                    startangle=90, textprops={"fontsize": 11},
                )
            ax.set_title(f"Audit Result\n(n={total_with_valid})", fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()

            # Accuracy metric
            n_pass = result_counts["Pass"]
            n_fail = result_counts["Fail"]
            denom = n_pass + n_fail
            if denom > 0:
                accuracy = n_pass / denom * 100
                display(HTML(
                    f"<h4>Accuracy</h4>"
                    f"<p>Markets where all audits agree with answer: <b>{n_pass}</b><br>"
                    f"Markets where no audit agrees with answer: <b>{n_fail}</b><br>"
                    f"Accuracy (pass / (pass + fail)): <b>{accuracy:.1f}%</b> ({n_pass}/{denom})</p>"
                    f"<p><i>Excludes markets with partial agreement and markets with no valid audits.</i></p>"
                ))

### Challenge analysis

Markets where the initial answer was disputed via Realitio's bond escalation mechanism.
A market is considered **challenged** if it has more than one distinct answer submission or the current bond exceeds the initial bond (0.001 xDAI).

In [ ]:
from omen_markets import (
    INVALID_ANSWER_HEX as _INVALID_HEX,
    get_market_state as _get_state,
    get_market_current_answer as _get_mkt_answer,
)

_INITIAL_BOND_WEI = 1_000_000_000_000_000  # 0.001 xDAI

def _is_challenged(market):
    """A market is challenged if bond escalated or multiple distinct answers exist."""
    question = market.get("question") or {}
    answers = question.get("answers") or []
    bond = int(market.get("currentAnswerBond") or question.get("currentAnswerBond") or 0)
    return len(answers) > 1 or bond > _INITIAL_BOND_WEI

def _n_challenges(market):
    """Number of challenges: distinct answers beyond the first, minimum 1 for bond-only escalation."""
    answers = (market.get("question") or {}).get("answers") or []
    return max(len(answers) - 1, 1)

def _answer_hex_to_label(hex_val, outcomes):
    """Convert hex answer to human-readable label."""
    if not hex_val:
        return "N/A"
    if hex_val.lower() == _INVALID_HEX.lower():
        return "Invalid"
    try:
        idx = int(hex_val, 16)
        return outcomes[idx] if idx < len(outcomes) else f"Index {idx}"
    except (ValueError, IndexError):
        return hex_val[:10]

# Collect all closed markets across creators
_all_closed = []
for key, markets in markets_by_creator.items():
    name = MARKET_CREATORS[key]["name"]
    for m in markets:
        if str(_get_state(m)) != "Closed":
            continue
        _all_closed.append((name, m))

_challenged = [(name, m) for name, m in _all_closed if _is_challenged(m)]
_not_challenged = [(name, m) for name, m in _all_closed if not _is_challenged(m)]

n_closed = len(_all_closed)
n_challenged = len(_challenged)

# --- Overview ---
display(HTML("<h4>Challenge overview</h4>"))
overview_rows = [
    {"Metric": "Total closed markets", "Value": n_closed},
    {"Metric": "Challenged", "Value": f"{n_challenged} ({n_challenged/n_closed*100:.1f}%)" if n_closed else 0},
    {"Metric": "Not challenged", "Value": f"{n_closed - n_challenged} ({(n_closed - n_challenged)/n_closed*100:.1f}%)" if n_closed else 0},
]
display(HTML(pd.DataFrame(overview_rows).to_html(escape=False, index=False)))

if n_challenged > 0:
    # --- Per-creator breakdown ---
    display(HTML("<h4>Challenged markets by creator</h4>"))
    creator_rows = []
    for key in MARKET_CREATORS:
        name = MARKET_CREATORS[key]["name"]
        total_creator = sum(1 for n, _ in _all_closed if n == name)
        chall_creator = sum(1 for n, _ in _challenged if n == name)
        if total_creator == 0:
            continue
        creator_rows.append({
            "Creator": name,
            "Closed": total_creator,
            "Challenged": chall_creator,
            "% Challenged": f"{chall_creator/total_creator*100:.1f}%",
        })
    display(HTML(pd.DataFrame(creator_rows).to_html(escape=False, index=False)))

    # --- Number of challenges per market ---
    display(HTML("<h4>Number of challenges per market</h4>"))
    display(HTML("<p><i>Each distinct answer beyond the first counts as a challenge. "
                 "Bond-only escalation (same answer, higher bond) counts as 1.</i></p>"))
    chall_counts = [_n_challenges(m) for _, m in _challenged]
    chall_counter = Counter(chall_counts)
    chall_rows = []
    for n_ch in sorted(chall_counter.keys()):
        count = chall_counter[n_ch]
        chall_rows.append({
            "# Challenges": n_ch,
            "Markets": count,
            "%": f"{count/n_challenged*100:.1f}%",
        })
    display(HTML(pd.DataFrame(chall_rows).to_html(escape=False, index=False)))

    # --- Did the challenge flip the answer? ---
    # Compare: first answer submitted vs final answer
    display(HTML("<h4>Challenge outcome — did the answer flip?</h4>"))
    flipped = 0
    stayed = 0
    invalid_final = 0
    for _, m in _challenged:
        question = m.get("question") or {}
        answers = question.get("answers") or []
        current = (m.get("currentAnswer") or "").lower()

        if current == _INVALID_HEX.lower():
            invalid_final += 1
            continue

        if len(answers) < 2:
            # Bond escalated but only 1 distinct answer value — answer stayed
            stayed += 1
            continue

        # The first answer in the list is the earliest submission
        # If current answer differs from answers[0], the challenge flipped it
        first_answer = (answers[0].get("answer") or "").lower()
        if current != first_answer:
            flipped += 1
        else:
            stayed += 1

    outcome_rows = [
        {"Outcome": "Answer stayed (original won)", "Markets": stayed,
         "%": f"{stayed/n_challenged*100:.1f}%"},
        {"Outcome": "Answer flipped (challenger won)", "Markets": flipped,
         "%": f"{flipped/n_challenged*100:.1f}%"},
        {"Outcome": "Resolved as Invalid", "Markets": invalid_final,
         "%": f"{invalid_final/n_challenged*100:.1f}%"},
    ]
    display(HTML(pd.DataFrame(outcome_rows).to_html(escape=False, index=False)))

    # Pie chart
    outcome_labels = []
    outcome_values = []
    outcome_colors_map = {"Answer stayed": "#4CAF50", "Answer flipped": "#F44336", "Invalid": "#9E9E9E"}
    for label, val, color_key in [
        ("Answer stayed", stayed, "Answer stayed"),
        ("Answer flipped", flipped, "Answer flipped"),
        ("Invalid", invalid_final, "Invalid"),
    ]:
        if val > 0:
            outcome_labels.append(label)
            outcome_values.append(val)

    if outcome_values:
        fig, ax = plt.subplots(figsize=(6, 5))
        oc_colors = [outcome_colors_map[l] for l in outcome_labels]
        total_oc = sum(outcome_values)
        ax.pie(
            outcome_values, labels=outcome_labels, colors=oc_colors,
            autopct=lambda pct, t=total_oc: f"{pct:.1f}%\n({int(round(pct * t / 100))})",
            startangle=90, textprops={"fontsize": 11},
        )
        ax.set_title(f"Challenge Outcome\n(n={n_challenged})", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

    # --- Bond escalation ---
    display(HTML("<h4>Bond escalation</h4>"))
    bonds_xdai = []
    for _, m in _challenged:
        bond_wei = int(m.get("currentAnswerBond") or (m.get("question") or {}).get("currentAnswerBond") or 0)
        bonds_xdai.append(bond_wei / 1e18)

    bond_stats = {
        "Metric": ["Min", "Median", "Mean", "Max", "Total"],
        "Bond (xDAI)": [
            f"{min(bonds_xdai):.4f}",
            f"{sorted(bonds_xdai)[len(bonds_xdai)//2]:.4f}",
            f"{sum(bonds_xdai)/len(bonds_xdai):.4f}",
            f"{max(bonds_xdai):.4f}",
            f"{sum(bonds_xdai):.4f}",
        ],
    }
    display(HTML(pd.DataFrame(bond_stats).to_html(escape=False, index=False)))

    # --- Cross-reference with audit data ---
    if _audit_data:
        display(HTML("<h4>Challenged markets vs audit agreement</h4>"))
        chall_pass = chall_partial = chall_fail = chall_no_audit = 0
        unchall_pass = unchall_partial = unchall_fail = unchall_no_audit = 0

        def _audit_result(market):
            audits = _audit_data.get(market.get("id", ""))
            if not audits:
                return "no_audit"
            answer = _get_mkt_answer(market).lower()
            outcomes = [o.lower() for o in (market.get("question") or {}).get("outcomes", [])]
            matching = valid = 0
            for a in audits.values():
                if not isinstance(a, dict):
                    continue
                resp = (a.get("response") or {}).get("answer")
                if not isinstance(resp, str):
                    continue
                if resp.lower() not in outcomes:
                    continue
                valid += 1
                if resp.lower() == answer:
                    matching += 1
            if valid == 0:
                return "no_audit"
            if matching == valid:
                return "pass"
            elif matching > 0:
                return "partial"
            return "fail"

        for _, m in _challenged:
            r = _audit_result(m)
            if r == "pass": chall_pass += 1
            elif r == "partial": chall_partial += 1
            elif r == "fail": chall_fail += 1
            else: chall_no_audit += 1

        for _, m in _not_challenged:
            r = _audit_result(m)
            if r == "pass": unchall_pass += 1
            elif r == "partial": unchall_partial += 1
            elif r == "fail": unchall_fail += 1
            else: unchall_no_audit += 1

        def _pct(val, total):
            return f"{val} ({val/total*100:.1f}%)" if total > 0 else "0"

        n_chall_audited = chall_pass + chall_partial + chall_fail
        n_unchall_audited = unchall_pass + unchall_partial + unchall_fail

        cross_rows = [
            {
                "": "Challenged",
                "Audited": n_chall_audited,
                "Pass": _pct(chall_pass, n_chall_audited),
                "Partial": _pct(chall_partial, n_chall_audited),
                "Fail": _pct(chall_fail, n_chall_audited),
            },
            {
                "": "Not challenged",
                "Audited": n_unchall_audited,
                "Pass": _pct(unchall_pass, n_unchall_audited),
                "Partial": _pct(unchall_partial, n_unchall_audited),
                "Fail": _pct(unchall_fail, n_unchall_audited),
            },
        ]
        display(HTML(pd.DataFrame(cross_rows).to_html(escape=False, index=False)))
        display(HTML(
            "<p><i>Pass = all valid audits agree with market answer. "
            "Fail = no audit agrees. Higher fail rate among challenged markets "
            "suggests challenges correlate with incorrect initial answers.</i></p>"
        ))

In [ ]:
# Collect flipped markets: challenge changed the final answer
_flipped_markets = []
for name, m in _challenged:
    question = m.get("question") or {}
    answers = question.get("answers") or []
    current = (m.get("currentAnswer") or "").lower()

    if current == _INVALID_HEX.lower():
        continue
    if len(answers) < 2:
        continue

    first_answer = (answers[0].get("answer") or "").lower()
    if current != first_answer:
        _flipped_markets.append((name, m))

# Sort by answer timestamp descending
_flipped_markets.sort(
    key=lambda x: int((x[1].get("question") or {}).get("currentAnswerTimestamp") or x[1].get("creationTimestamp", "0")),
    reverse=True,
)

_flip_rows = []
for name, m in _flipped_markets:
    question = m.get("question") or {}
    answers = question.get("answers") or []
    outcomes = question.get("outcomes", [])
    title_text = (question.get("title", ""))[:80]
    presagio_url = f'https://presagio.pages.dev/markets?id={m.get("id", "")}'

    first_answer = (answers[0].get("answer") or "").lower() if answers else ""
    first_label = _answer_hex_to_label(first_answer, outcomes) if first_answer else "--"
    final_label = _get_mkt_answer(m)

    bond_wei = int(m.get("currentAnswerBond") or question.get("currentAnswerBond") or 0)
    bond_xdai = bond_wei / 1e18

    # Audit info
    audit_str = ""
    market_audits = _audit_data.get(m.get("id", ""))
    if market_audits:
        a_matching, a_valid, a_total = _audit_stats(m, market_audits)
        if a_total > 0:
            if a_valid == 0:
                audit_str = f"{a_matching}/{a_valid}/{a_total}"
            elif a_matching == a_valid:
                audit_str = f"{a_matching}/{a_valid}/{a_total} \U0001f7e2"
            elif a_matching > 0:
                audit_str = f"{a_matching}/{a_valid}/{a_total} \U0001f7e1"
            else:
                audit_str = f"{a_matching}/{a_valid}/{a_total} \U0001f534"

    _flip_rows.append({
        "Title": f'<a href="{presagio_url}" target="_blank">{title_text}</a>',
        "Creator": name,
        "# Answers": len(answers),
        "First answer": first_label,
        "Final answer": final_label,
        "Bond (xDAI)": f"{bond_xdai:.4f}",
        "Audit": audit_str,
        "Answered (UTC)": datetime.fromtimestamp(
            int(question.get("currentAnswerTimestamp") or m.get("creationTimestamp", 0)),
            tz=timezone.utc,
        ).strftime("%Y-%m-%d %H:%M"),
    })

display(HTML(f"<h4>Flipped markets ({len(_flipped_markets)})</h4>"))
if _flip_rows:
    display(HTML(pd.DataFrame(_flip_rows).to_html(escape=False, index=False)))
else:
    display(HTML("<p><i>No flipped markets found.</i></p>"))

## Mech Requests

Fetch all mech requests made by each market creator and their associated delivers.

The market creator uses the Realitio **question ID as the nonce** for mech requests.
When a response is `not_determinable`, it retries with the same nonce but the subgraph
assigns a new request ID each time.

Data is cached to `notebooks/.cache/mech_requests_<key>.json` — already-delivered
requests with IPFS contents won't be re-fetched.

In [ ]:
from mech_requests import fetch_mech_requests

mech_data = {
    key: fetch_mech_requests(cfg["safe_contract_address"])
    for key, cfg in MARKET_CREATORS.items()
}

In [ ]:
import json
from collections import Counter

for key, reqs in mech_data.items():
    cfg = MARKET_CREATORS[key]
    name = cfg["name"]

    total = len(reqs)
    with_deliver = sum(1 for r in reqs.values() if "deliver" in r)
    without_deliver = total - with_deliver

    # Count unique nonces (= unique markets) and retries
    nonce_counts = Counter()
    for r in reqs.values():
        try:
            content = json.loads(r["parsedRequest"]["content"])
            nonce_counts[content.get("nonce", "")] += 1
        except (json.JSONDecodeError, KeyError, TypeError):
            pass
    unique_markets = len(nonce_counts)
    retried = sum(1 for c in nonce_counts.values() if c > 1)

    display(HTML(f"<h4>{name}</h4>"))
    display(HTML(
        f"<ul>"
        f"<li>Total requests: <b>{total}</b></li>"
        f"<li>With delivery: <b>{with_deliver}</b></li>"
        f"<li>Without delivery: <b>{without_deliver}</b></li>"
        f"<li>Unique markets (nonces): <b>{unique_markets}</b></li>"
        f"<li>Markets with retries: <b>{retried}</b></li>"
        f"</ul>"
    ))

### Mech result distribution

Distribution of mech deliver results, categorized by the first key-value pair of the JSON response, or **Error** if the response is not valid JSON.

In [ ]:
def _categorize_mech_result(result_str: str) -> str:
    """Categorize a mech deliver result into a human-readable label."""
    try:
        parsed = json.loads(result_str)
        if isinstance(parsed, dict) and parsed:
            first_key = list(parsed.keys())[0]
            first_val = parsed[first_key]
            return f"{first_key}={first_val}"
        return "Valid (other)"
    except (json.JSONDecodeError, TypeError, ValueError):
        return "Error"

RESULT_COLORS = {
    "has_occurred=True": "#4CAF50",
    "has_occurred=False": "#F44336",
    "has_occurred=None": "#BDBDBD",
    "is_determinable=False": "#FF9800",
    "is_valid=False": "#9E9E9E",
    "Error": "#B71C1C",
}


def _plot_mech_result_distribution(mech_data, creators, subtitle, cutoff_ts=None):
    """Plot mech result distribution pie charts. If cutoff_ts is None, include all requests."""
    mech_dist = {}
    for key, reqs in mech_data.items():
        name = creators[key]["name"]
        counts = Counter()
        for r in reqs.values():
            if cutoff_ts is not None and int(r.get("blockTimestamp", 0)) < cutoff_ts:
                continue
            if "deliver" not in r or "ipfsContents" not in r.get("deliver", {}):
                counts["No delivery"] += 1
                continue
            result_str = str(r["deliver"]["ipfsContents"].get("result", ""))
            counts[_categorize_mech_result(result_str)] += 1
        mech_dist[name] = counts

    n_creators = len(mech_dist)
    fig, axes = plt.subplots(1, max(n_creators, 1), figsize=(6 * max(n_creators, 1), 5))
    if n_creators == 1:
        axes = [axes]

    for ax, (name, counts) in zip(axes, mech_dist.items()):
        if not counts:
            ax.set_title(f"{name}\n(no requests)")
            ax.axis("off")
            continue
        labels = list(counts.keys())
        values = list(counts.values())
        colors = [RESULT_COLORS.get(l, "#607D8B") for l in labels]
        total = sum(values)
        ax.pie(
            values, labels=labels, colors=colors,
            autopct=lambda pct, t=total: f"{pct:.1f}%\n({int(round(pct * t / 100))})",
            startangle=90, textprops={"fontsize": 10},
        )
        ax.set_title(f"{name}\n(n={total})", fontsize=12, fontweight="bold")

    fig.suptitle(f"Mech Result Distribution\n{subtitle}", fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout()
    plt.show()


cutoff_ts = int((now_utc - timedelta(days=STATS_PERIOD_DAYS)).timestamp())

# Study period
_plot_mech_result_distribution(mech_data, MARKET_CREATORS, f"Requests {_period_label}", cutoff_ts=cutoff_ts)

# Overall
_plot_mech_result_distribution(mech_data, MARKET_CREATORS, "All requests")

### Mech requests per market

Number of mech requests (including retries) associated with each closed market, matched via the Realitio question ID (= mech nonce).

In [ ]:
from collections import defaultdict as _defaultdict
from omen_markets import get_market_state as _get_mkt_state

def _build_nonce_idx(reqs):
    """Build nonce -> list of requests index."""
    idx = _defaultdict(list)
    for r in reqs.values():
        try:
            content = json.loads(r["parsedRequest"]["content"])
            nonce = content.get("nonce", "")
            if nonce:
                idx[nonce].append(r)
        except (json.JSONDecodeError, KeyError, TypeError):
            pass
    return idx


def _mech_counts_per_market(markets, nonce_idx, cutoff_ts=None):
    """Return list of mech request counts per closed market."""
    counts = []
    for m in markets:
        if str(_get_mkt_state(m)) != "Closed":
            continue
        if cutoff_ts is not None:
            answer_ts = int((m.get("question") or {}).get("currentAnswerTimestamp") or 0)
            if answer_ts < cutoff_ts:
                continue
        question_id = (m.get("question") or {}).get("id", "")
        n_reqs = len(nonce_idx.get(question_id, []))
        counts.append(n_reqs)
    return counts


def _display_mech_per_market(creators_config, markets_by_creator, mech_data, subtitle, cutoff_ts=None):
    """Display table + bar chart of mech requests per closed market."""
    all_counts_by_creator = {}
    for key in creators_config:
        name = creators_config[key]["name"]
        markets = markets_by_creator.get(key, [])
        reqs = mech_data.get(key, {})
        nonce_idx = _build_nonce_idx(reqs)
        counts = _mech_counts_per_market(markets, nonce_idx, cutoff_ts=cutoff_ts)
        all_counts_by_creator[name] = counts

    # Summary table
    all_values = sorted(set(v for vals in all_counts_by_creator.values() for v in vals))
    if not all_values:
        display(HTML(f"<h4>Mech requests per closed market — {subtitle}</h4>"))
        display(HTML("<p><i>No closed markets.</i></p>"))
        return

    table_rows = []
    for name, counts in all_counts_by_creator.items():
        total = len(counts)
        if total == 0:
            continue
        counter = Counter(counts)
        row = {"Creator": f"{name} (n={total})"}
        for v in all_values:
            c = counter.get(v, 0)
            pct = c / total * 100
            row[f"{v} req{'s' if v != 1 else ''}"] = f"{c} ({pct:.1f}%)"
        table_rows.append(row)

    display(HTML(f"<h4>Mech requests per closed market — {subtitle}</h4>"))
    display(HTML(pd.DataFrame(table_rows).fillna("").to_html(escape=False, index=False)))

    # Bar chart per creator
    n_creators = len(all_counts_by_creator)
    fig, axes = plt.subplots(1, max(n_creators, 1), figsize=(6 * max(n_creators, 1), 4))
    if n_creators == 1:
        axes = [axes]

    for ax, (name, counts) in zip(axes, all_counts_by_creator.items()):
        if not counts:
            ax.set_title(f"{name}\n(no data)")
            ax.axis("off")
            continue
        counter = Counter(counts)
        x = sorted(counter.keys())
        y = [counter[v] for v in x]
        total = sum(y)
        ax.bar(x, y, color="steelblue", edgecolor="black", alpha=0.8)
        ax.set_xlabel("# Mech requests")
        ax.set_ylabel("# Markets")
        ax.set_xticks(x)
        ax.grid(True, alpha=0.3, axis="y")
        ax.set_title(f"{name}\n(n={total})", fontsize=12, fontweight="bold")

    fig.suptitle(f"Mech Requests per Closed Market\n{subtitle}", fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout()
    plt.show()


# Study period
_display_mech_per_market(
    MARKET_CREATORS, markets_by_creator, mech_data,
    f"Markets closed {_period_label}",
    cutoff_ts=cutoff_ts,
)

# Overall
_display_mech_per_market(
    MARKET_CREATORS, markets_by_creator, mech_data,
    "All closed markets",
)

### Mech deliver delay

Time between the mech request and its delivery, measured in **blocks** and **seconds** (`deliver - request`).

In [ ]:
import numpy as np
from scipy import stats

def _plot_delay_histogram(axes_row, mech_data, creators, value_fn, xlabel, unit, cutoff_ts):
    """Plot deliver delay histograms for each creator (only requests within period)."""
    for ax, (key, reqs) in zip(axes_row, mech_data.items()):
        name = creators[key]["name"]

        delays = []
        for r in reqs.values():
            if int(r.get("blockTimestamp", 0)) < cutoff_ts:
                continue
            if "deliver" in r:
                delay = value_fn(r)
                if delay >= 0:
                    delays.append(delay)

        if not delays:
            ax.set_title(f"{name}\n(no delivers)")
            ax.axis("off")
            continue

        desc = stats.describe(delays)
        print(f"{name} ({unit}): n={desc.nobs}, mean={desc.mean:.1f}, var={desc.variance:.1f}, "
              f"min={desc.minmax[0]}, max={desc.minmax[1]}")

        mean = desc.mean
        std = np.sqrt(desc.variance)
        filtered = [d for d in delays if abs(d - mean) <= 5 * std]
        n_outliers = len(delays) - len(filtered)
        if n_outliers > 0:
            print(f"  Filtered {n_outliers} outliers (>5σ)")

        ax.hist(filtered, bins=100, color="steelblue", edgecolor="black", alpha=0.8)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Frequency")

        median_d = float(np.median(filtered))
        mean_d = float(np.mean(filtered))
        ax.axvline(median_d, color="red", linestyle="--", linewidth=1.5, label=f"Median: {median_d:.0f}")
        ax.axvline(mean_d, color="orange", linestyle="--", linewidth=1.5, label=f"Mean: {mean_d:.0f}")
        ax.legend()
        ax.set_title(f"{name}\n(n={len(filtered)})", fontsize=12, fontweight="bold")
        ax.grid(True, alpha=0.3)

n_creators = max(len(mech_data), 1)
fig, axes = plt.subplots(2, n_creators, figsize=(8 * n_creators, 10))
if n_creators == 1:
    axes = axes.reshape(2, 1)

_plot_delay_histogram(
    axes[0], mech_data, MARKET_CREATORS,
    value_fn=lambda r: int(r["deliver"]["blockNumber"]) - int(r["blockNumber"]),
    xlabel="Deliver delay (blocks)", unit="blocks", cutoff_ts=cutoff_ts,
)
_plot_delay_histogram(
    axes[1], mech_data, MARKET_CREATORS,
    value_fn=lambda r: int(r["deliver"]["blockTimestamp"]) - int(r["blockTimestamp"]),
    xlabel="Deliver delay (seconds)", unit="seconds", cutoff_ts=cutoff_ts,
)

fig.suptitle(f"Mech Deliver Delay\nRequests {_period_label}", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### Latest mech requests

Most recent requests with their associated responses. Request and deliver IDs link to their IPFS contents.

In [ ]:
from mech_requests import get_request_ipfs_url, get_deliver_ipfs_url

LATEST_MECH_REQUESTS = 20

def _is_valid_response(response: str) -> bool:
    """A response is valid if it parses as JSON."""
    try:
        json.loads(response)
        return True
    except (json.JSONDecodeError, TypeError, ValueError):
        return False

for key, reqs in mech_data.items():
    name = MARKET_CREATORS[key]["name"]

    # Sort by blockTimestamp descending
    sorted_reqs = sorted(reqs.values(), key=lambda r: int(r["blockTimestamp"]), reverse=True)

    rows = []
    for r in sorted_reqs[:LATEST_MECH_REQUESTS]:
        req_ts = datetime.fromtimestamp(int(r["blockTimestamp"]), tz=timezone.utc)
        req_url = get_request_ipfs_url(r)
        req_id_short = r["id"][:18] + "..."
        req_link = f'<a href="{req_url}" target="_blank">{req_id_short}</a>'

        # Extract question from IPFS contents
        question = ""
        if "ipfsContents" in r:
            question = r["ipfsContents"].get("prompt", "")

        # Deliver info
        deliver_link = ""
        deliver_ts_str = ""
        response_display = ""
        if "deliver" in r:
            d = r["deliver"]
            deliver_ts = datetime.fromtimestamp(int(d["blockTimestamp"]), tz=timezone.utc)
            deliver_ts_str = deliver_ts.strftime("%Y-%m-%d %H:%M")
            deliver_url = get_deliver_ipfs_url(d)
            deliver_id_short = d["id"][:18] + "..."
            if deliver_url:
                deliver_link = f'<a href="{deliver_url}" target="_blank">{deliver_id_short}</a>'
            else:
                deliver_link = deliver_id_short
            if "ipfsContents" in d:
                raw_response = str(d["ipfsContents"].get("result", ""))
                truncated = raw_response[:100] + ("..." if len(raw_response) > 100 else "")
                if _is_valid_response(raw_response):
                    response_display = truncated
                else:
                    response_display = f"{truncated} \U0001f534"

        rows.append({
            "Request ID": req_link,
            "Request Time (UTC)": req_ts.strftime("%Y-%m-%d %H:%M"),
            "Question": question[:120] + ("..." if len(question) > 120 else ""),
            "Deliver ID": deliver_link,
            "Deliver Time (UTC)": deliver_ts_str,
            "Response": response_display,
        })

    df_latest = pd.DataFrame(rows)
    display(HTML(f"<h4>{name} — last {LATEST_MECH_REQUESTS} requests</h4>"))
    display(HTML(df_latest.to_html(escape=False, index=False)))

## Markets + Mech Requests

Join markets with their associated mech requests via the Realitio question ID (= mech nonce).
Each market may have multiple mech requests (retries after `not_determinable`).

Use the widget below to filter by market state.

In [ ]:
from collections import defaultdict, Counter
from omen_markets import get_market_state, get_market_current_answer
from mech_requests import get_deliver_ipfs_url

# Configure which states to display and max rows
MARKET_TABLE_STATES = [
    MarketState.OPEN,
    MarketState.PENDING,
    MarketState.FINALIZING,
    MarketState.ARBITRATING,
    # MarketState.CLOSED,
    MarketState.UNKNOWN,
]
MAX_MARKETS_DISPLAYED = 200

# Initial bond set by the market creator (0.001 xDAI)
INITIAL_BOND_WEI = 1_000_000_000_000_000

# Load audit data (generated externally via fpmm_audits.py)
_AUDITS_JSON_PATH = "fpmm_audits.json"
try:
    with open(_AUDITS_JSON_PATH, "r", encoding="UTF-8") as _f:
        _all_audits = json.load(_f).get("fpmmAudits", {})
except FileNotFoundError:
    _all_audits = {}


def _get_matching_audits(market, audits):
    """Get number of matching, valid, and total audits for a market."""
    market_answer = get_market_current_answer(market).lower()
    outcomes = [o.lower() for o in (market.get("question") or {}).get("outcomes", [])]
    total = 0
    matching = 0
    valid = 0
    for audit in audits.values():
        if not isinstance(audit, dict):
            continue
        audit_answer = (audit.get("response") or {}).get("answer")
        if not isinstance(audit_answer, str):
            continue
        total += 1
        if audit_answer.lower() not in outcomes:
            continue
        valid += 1
        if market_answer == audit_answer.lower():
            matching += 1
    return matching, valid, total


def _build_nonce_index(reqs):
    index = defaultdict(list)
    for r in reqs.values():
        try:
            content = json.loads(r["parsedRequest"]["content"])
            nonce = content.get("nonce", "")
            if nonce:
                index[nonce].append(r)
        except (json.JSONDecodeError, KeyError, TypeError):
            pass
    for nonce in index:
        index[nonce].sort(key=lambda r: int(r["blockTimestamp"]))
    return index

def _format_delivers(mech_reqs):
    n = len(mech_reqs)
    parts = []
    for i, r in enumerate(reversed(mech_reqs), 0):
        label = n - i
        if "deliver" in r:
            url = get_deliver_ipfs_url(r["deliver"])
            parts.append(f'<a href="{url}" target="_blank">[{label}]</a>' if url else f"[{label}]")
        else:
            parts.append("[X]")
    return "".join(parts)

def _format_mech_response(mech_reqs):
    for r in reversed(mech_reqs):
        if "deliver" in r and "ipfsContents" in r.get("deliver", {}):
            raw = str(r["deliver"]["ipfsContents"].get("result", ""))
            return _categorize_mech_result(raw)
    return ""

def _format_state(market):
    s = get_market_state(market)
    return f"{s} {MARKET_STATE_ICONS.get(s, '')}"

def _format_remaining_time(market):
    if get_market_state(market) != MarketState.FINALIZING:
        return "--"
    remaining = int(market.get("answerFinalizedTimestamp", 0)) - int(now_utc.timestamp())
    if remaining <= 0:
        return "00:00"
    return f"{remaining // 3600:02d}:{(remaining % 3600) // 60:02d}"

def _format_ts(market, field):
    ts = market.get(field)
    if not ts:
        return "--"
    return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%d %H:%M")

def _format_answers_info(market):
    """Format number of answer submissions with challenge indicator."""
    question = market.get("question") or {}
    answers = question.get("answers") or []
    n_distinct = len(answers)
    if n_distinct == 0:
        return "--"
    current_bond = int(market.get("currentAnswerBond") or question.get("currentAnswerBond") or 0)
    was_challenged = current_bond > INITIAL_BOND_WEI or n_distinct > 1
    label = str(n_distinct)
    if was_challenged:
        label += " \U0001f534"
    return label

def _format_audit_info(market):
    """Format audit info as 'matching/valid/total icon' in a single string.

    Icon: green = all valid match, yellow = some match, red = none match.
    Returns empty string if market has no audits.
    """
    market_audits = _all_audits.get(market.get("id", ""))
    if not market_audits:
        return ""

    matching, valid, total = _get_matching_audits(market, market_audits)
    if total == 0:
        return ""

    if valid == 0:
        return f"{matching}/{valid}/{total}"

    if matching == valid:
        icon = "\U0001f7e2"  # green
    elif matching > 0:
        icon = "\U0001f7e1"  # yellow
    else:
        icon = "\U0001f534"  # red

    return f"{matching}/{valid}/{total} {icon}"

# State summary table with icons in headers
df_states_display = df_states.rename(
    columns={col: f"{col} {MARKET_STATE_ICONS.get(MarketState.__members__.get(col.upper()), '')}" for col in df_states.columns}
)
display(HTML(df_states_display.to_html()))

state_set = set(MARKET_TABLE_STATES)

for key in MARKET_CREATORS:
    name = MARKET_CREATORS[key]["name"]
    safe_addr = MARKET_CREATORS[key]["safe_contract_address"].lower()
    markets = markets_by_creator.get(key, [])
    reqs = mech_data.get(key, {})
    nonce_index = _build_nonce_index(reqs)

    filtered = sorted(
        [m for m in markets if get_market_state(m) in state_set],
        key=lambda m: int(m["creationTimestamp"]),
        reverse=True,
    )
    total = len(filtered)

    # Count per state for header with icons
    state_counts = Counter(get_market_state(m) for m in filtered)
    states_detail = ", ".join(
        f"{state_counts.get(s, 0)} {s.name.capitalize()} {MARKET_STATE_ICONS.get(s, '')}"
        for s in MARKET_TABLE_STATES
        if state_counts.get(s, 0) > 0
    )

    filtered = filtered[:MAX_MARKETS_DISPLAYED]

    rows = []
    for m in filtered:
        question = m.get("question") or {}
        question_id = question.get("id", "")
        title_text = (question.get("title", ""))[:80]
        presagio_url = f'https://presagio.pages.dev/markets?id={m.get("id", "")}'
        mech_reqs = nonce_index.get(question_id, [])

        avg_delay = ""
        if mech_reqs:
            delays = [int(r["deliver"]["blockTimestamp"]) - int(r["blockTimestamp"])
                      for r in mech_reqs if "deliver" in r]
            delays = [d for d in delays if d >= 0]
            if delays:
                avg_delay = f"{sum(delays) / len(delays):.0f}s"

        rows.append({
            "Title": f'<a href="{presagio_url}" target="_blank">{title_text}</a>',
            "Vol (xDAI)": f"{float(m.get('collateralVolume', 0)) / 1e18:.2f}",
            "Answer": get_market_current_answer(m),
            "# Ans": _format_answers_info(m),
            "State": _format_state(m),
            "Finalizes in": _format_remaining_time(m),
            "Created (UTC)": _format_ts(m, "creationTimestamp"),
            "Resolved (UTC)": _format_ts(m, "resolutionTimestamp"),
            "Finalized (UTC)": _format_ts(m, "answerFinalizedTimestamp"),
            "Audit": _format_audit_info(m),
            "# Mech": len(mech_reqs),
            "Delivers": _format_delivers(mech_reqs) if mech_reqs else "",
            "Delay": avg_delay,
            "Last Mech Response": _format_mech_response(mech_reqs) if mech_reqs else "",
        })

    truncated = f" (showing {MAX_MARKETS_DISPLAYED} of {total})" if total > MAX_MARKETS_DISPLAYED else ""
    display(HTML(f"<h4>{name} — {total} markets ({states_detail}){truncated}</h4>"))
    if rows:
        display(HTML(pd.DataFrame(rows).to_html(escape=False, index=False)))
    else:
        display(HTML("<p><i>No markets match the selected states.</i></p>"))